# OpenSplat — Linux / CUDA resource-aware optimization

This notebook builds OpenSplat, profiles it across a **2 → 128 image** ladder, and
validates the **resource-aware process contract** that lets one binary run well from a
**2 GB desktop** to an **80 GB server** — choosing the host image store, a VRAM-bounded
gaussian cap, and a quality-guarded downscale automatically, with full CLI / env / JSON
override.

Everything here is **measured, not estimated**. The harness samples real `/proc` RSS and
`nvidia-smi` VRAM while training runs. Findings and Mermaid diagrams live in
`linux_research/` (gitignored).

**Sections**
1. Environment & hardware detection
2. Build (CUDA, against the in-tree pip libtorch)
3. Dataset + the N-image scaling ladder
4. The resource contract (auto / 2gb / 4gb / 6gb / 8gb / full-throttle)
5. RAM: baseline (f32) vs optimized (u8) image store
6. VRAM: the `--max-splats` cap
7. Tests (unit + regression + integration, incl. CUDA)
8. Summary

## 1. Environment & hardware detection
What machine are we on? The contract keys off exactly these numbers.

In [1]:
import os, subprocess, json, pathlib, textwrap
REPO = pathlib.Path('/content/OpenSplat')
os.chdir(REPO)

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)

print("=== GPU ==="); print(sh("nvidia-smi --query-gpu=name,memory.total,memory.free,driver_version --format=csv,noheader").stdout or "no GPU")
print("=== CPU/RAM ==="); print("cores:", os.cpu_count())
print(sh("free -h | head -2").stdout)
try:
    import torch
    print("torch", torch.__version__, "| cuda", torch.version.cuda, "| is_available", torch.cuda.is_available())
except Exception as e:
    print("torch import:", e)

=== GPU ===
NVIDIA A100-SXM4-80GB, 81920 MiB, 81153 MiB, 580.82.07

=== CPU/RAM ===
cores: 12
               total        used        free      shared  buff/cache   available
Mem:           167Gi       1.5Gi       154Gi       2.0Mi        10Gi       164Gi



torch 2.11.0+cu128 | cuda 12.8 | is_available True


## 2. Build

OpenSplat is built **CUDA** against the libtorch that ships inside the Colab `torch` wheel
(no multi-GB download). Single SM arch (`sm_80`, A100) keeps the compile fast; production
builds should pass `-DCMAKE_CUDA_ARCHITECTURES="70;75;80;86;89"`. Re-runs are incremental.

In [2]:
import torch, os, pathlib, subprocess
REPO = pathlib.Path('/content/OpenSplat')
BIN = REPO/'output'/'opensplat'
torch_cmake = pathlib.Path(torch.__file__).parent/'share'/'cmake'

# OpenCV C++ dev headers (the Python cv2 wheel is not enough for the C++ build).
if subprocess.run("pkg-config --exists opencv4", shell=True).returncode != 0 and \
   not list(pathlib.Path('/usr').rglob('OpenCVConfig.cmake')):
    subprocess.run("apt-get install -y -q libopencv-dev", shell=True)

if not BIN.exists():
    print(">> configuring + building (first build ~a few min)...")
    subprocess.run(f"cmake -S {REPO} -B {REPO}/build -DCMAKE_BUILD_TYPE=Release "
                   f"-DGPU_RUNTIME=CUDA -DCMAKE_PREFIX_PATH={torch_cmake} "
                   f"-DCMAKE_CUDA_ARCHITECTURES=80 -DCMAKE_RUNTIME_OUTPUT_DIRECTORY={REPO}/output",
                   shell=True, check=True)
    subprocess.run(f"cmake --build {REPO}/build -j{os.cpu_count()} --target opensplat", shell=True, check=True)
else:
    print(">> reusing existing binary:", BIN)
print(subprocess.run([str(BIN), "--version"], text=True, capture_output=True).stdout.strip())

>> reusing existing binary: /content/OpenSplat/output/opensplat


1.1.5 (git commit 8b1a8ca)


## 3. Dataset + the N-image scaling ladder

We use **drjohnson** (Deep Blending, ~1.17 MP images, native `-d 1` — avoids the
over-downscaling divergence cliff), from the project dataset on Hugging Face. Valid
N-image COLMAP sub-scenes (`N = 2,4,8,16,32,64,128`) are produced by `scripts/make_chunks.py`
(it rewrites a consistent `sparse/0`, not just a file copy).

In [3]:
import subprocess, pathlib
REPO = pathlib.Path('/content/OpenSplat')
LADDER = [2,4,8,16,32,64,128]
scene = REPO/'data'/'db'/'drjohnson'
if not (scene/'sparse').exists():
    print(">> downloading drjohnson ...")
    from huggingface_hub import snapshot_download
    snapshot_download('alexmkwizu/gaussian_training_datasets', repo_type='dataset',
                      allow_patterns=['db/drjohnson/**'], local_dir=str(REPO/'data'))
if not (REPO/'data'/'chunks'/'drjohnson_2'/'sparse').exists():
    subprocess.run("pip install -q pycolmap", shell=True)
    subprocess.run(f"python3 scripts/make_chunks.py {scene} --chunks {' '.join(map(str,LADDER))} --out data/chunks",
                   shell=True, check=True)
for n in LADDER:
    d = REPO/'data'/'chunks'/f'drjohnson_{n}'/'images'
    print(f"drjohnson_{n}: {len(list(d.glob('*'))) if d.exists() else 0} images")

drjohnson_2: 2 images
drjohnson_4: 4 images
drjohnson_8: 8 images
drjohnson_16: 16 images
drjohnson_32: 32 images
drjohnson_64: 64 images
drjohnson_128: 128 images


## 4. The resource-aware process contract

At startup OpenSplat detects RAM/VRAM/CPU + scene size and resolves a **contract**:
which host image store (`f32`/`u8`), a VRAM-bounded gaussian cap, a quality-guarded
downscale. Precedence: **CLI > env `OPENSPLAT_*` > `--contract` JSON > named profile > auto**.
Below: the same scene resolved under several profiles (note how `image store` and
`max gaussians` change).

In [4]:
import subprocess, pathlib, re
BIN = pathlib.Path('/content/OpenSplat')/'output'/'opensplat'
DS = 'data/chunks/drjohnson_128'
def contract(args):
    out = subprocess.run(f"{BIN} {DS} -n 1 {args}", shell=True, text=True,
                         capture_output=True, cwd='/content/OpenSplat').stdout
    block = re.search(r"OpenSplat resource contract.*?={10,}\n(.*?)={10,}", out, re.S)
    return block.group(0) if block else out[:600]
for prof in ["", "--profile 2gb", "--profile 8gb", "--profile full-throttle",
             "--max-splats 50000 --image-store u8"]:
    print(f"\n----- args: '{prof or 'auto (default)'}' -----")
    print(contract(prof))


----- args: 'auto (default)' -----


OpenSplat resource contract ====================
host:   RAM 171061 MB total / 167564 MB avail | CPU 12 cores | CUDA | VRAM 81920 MB total / 81148 MB free
scene:  128 images @ 1332x876
profile:        auto  [default]
ram budget:     167564 MB  [auto]
vram budget:    81148 MB  [auto]
image store:    f32  [auto]  (est preload 1811 MB)
max gaussians:  35895377  [auto]  (~69064 MB VRAM at cap)
min render px:  400  [default]

----- args: '--profile 2gb' -----


OpenSplat resource contract ====================
host:   RAM 171061 MB total / 167554 MB avail | CPU 12 cores | CUDA | VRAM 81920 MB total / 81148 MB free
scene:  128 images @ 1332x876
profile:        2gb  [cli]
ram budget:     2048 MB  [profile]
vram budget:    81148 MB  [auto]
image store:    u8  [auto]  (est preload 113 MB)
max gaussians:  35895377  [auto]  (~69064 MB VRAM at cap)
min render px:  400  [default]
downscale:      2  [auto]

----- args: '--profile 8gb' -----


OpenSplat resource contract ====================
host:   RAM 171061 MB total / 167607 MB avail | CPU 12 cores | CUDA | VRAM 81920 MB total / 81148 MB free
scene:  128 images @ 1332x876
profile:        8gb  [cli]
ram budget:     8192 MB  [profile]
vram budget:    81148 MB  [auto]
image store:    f32  [auto]  (est preload 1811 MB)
max gaussians:  35895377  [auto]  (~69064 MB VRAM at cap)
min render px:  400  [default]

----- args: '--profile full-throttle' -----


OpenSplat resource contract ====================
host:   RAM 171061 MB total / 167608 MB avail | CPU 12 cores | CUDA | VRAM 81920 MB total / 81148 MB free
scene:  128 images @ 1332x876
profile:        full-throttle  [cli]
ram budget:     171061 MB  [profile]
vram budget:    81920 MB  [profile]
image store:    f32  [auto]  (est preload 1811 MB)
max gaussians:  unbounded  [profile]
min render px:  400  [default]

----- args: '--max-splats 50000 --image-store u8' -----


OpenSplat resource contract ====================
host:   RAM 171061 MB total / 167569 MB avail | CPU 12 cores | CUDA | VRAM 81920 MB total / 81148 MB free
scene:  128 images @ 1332x876
profile:        auto  [default]
ram budget:     167569 MB  [auto]
vram budget:    81148 MB  [auto]
image store:    u8  [cli]  (est preload 452 MB)
max gaussians:  50000  [cli]  (~695 MB VRAM at cap)
min render px:  400  [default]


### Override via a JSON contract file
A process-contract file lets ops pin policy without touching the command line.

In [5]:
import json, subprocess, pathlib, re
REPO = pathlib.Path('/content/OpenSplat')
cpath = REPO/'linux_research'/'demo_contract.json'
cpath.parent.mkdir(parents=True, exist_ok=True)
cpath.write_text(json.dumps({"profile":"4gb","image_store":"u8","max_gaussians":250000,"min_render_px":400}, indent=2))
out = subprocess.run(f"output/opensplat data/chunks/drjohnson_8 -n 1 --contract {cpath}",
                     shell=True, text=True, capture_output=True, cwd=REPO).stdout
print(re.search(r"OpenSplat resource contract.*?={10,}\n.*?={10,}", out, re.S).group(0))

OpenSplat resource contract ====================
host:   RAM 171061 MB total / 167526 MB avail | CPU 12 cores | CUDA | VRAM 81920 MB total / 81148 MB free
scene:  8 images @ 1332x876
profile:        4gb  [json]
ram budget:     4096 MB  [profile]
vram budget:    81148 MB  [auto]
image store:    u8  [json]  (est preload 28 MB)
max gaussians:  250000  [json]  (~1076 MB VRAM at cap)
min render px:  400  [json]


## 5. RAM — baseline (f32) vs optimized (u8) image store

OpenSplat decodes every training image to RAM up front. Storing **uint8** instead of
**float32** is 4× smaller on the dominant term and **bit-identical** (source JPEGs are
uint8), so it costs no quality. We measure peak RSS across the ladder with
`scripts/bench_resource.py`. Pre-measured JSONs are loaded if present; otherwise a quick
live sweep runs.

In [6]:
import json, subprocess, pathlib, os
REPO = pathlib.Path('/content/OpenSplat'); LADDER=[2,4,8,16,32,64,128]
def run_bench(ds, args, out, label):
    subprocess.run(f"python3 scripts/bench_resource.py --bin ./output/opensplat "
                   f"--dataset {ds} --extra-args '{args}' --label {label} --out {out} --poll-ms 40",
                   shell=True, cwd=REPO, capture_output=True)
def load_or_run(kind, store_args, subdir, n):
    p = REPO/'linux_research'/'measurements'/subdir/f'{"n" if subdir=="baseline" else "u8_n"}{n}.json'
    if not p.exists():
        p.parent.mkdir(parents=True, exist_ok=True)
        run_bench(f'data/chunks/drjohnson_{n}', f'-n 1000 {store_args}', p, f'{kind}_n{n}')
    return json.load(open(p)) if p.exists() else None
base = {n: load_or_run('baseline','--image-store f32','baseline',n) for n in LADDER}
opt  = {n: load_or_run('opt','--image-store u8','optimized',n) for n in LADDER}
print(f"{'N':>4} {'f32 RSS':>9} {'u8 RSS':>9} {'saved':>11} {'f32 it/s':>9} {'u8 it/s':>9}")
for n in LADDER:
    b,o = base[n], opt[n]
    if b and o:
        sv=b['peak_rss_mb']-o['peak_rss_mb']; pct=100*sv/b['peak_rss_mb']
        print(f"{n:>4} {b['peak_rss_mb']:>8.0f}M {o['peak_rss_mb']:>8.0f}M {sv:>6.0f}M {pct:>3.0f}% {b['iters_per_s']:>9.1f} {o['iters_per_s']:>9.1f}")

   N   f32 RSS    u8 RSS       saved  f32 it/s   u8 it/s
   2     1470M     1433M     37M   3%      92.1      88.9
   4     1445M     1481M    -36M  -2%      92.4      89.6
   8     1554M     1441M    113M   7%      91.7      88.0
  16     1733M     1632M    101M   6%      89.5      86.6
  32     2232M     1738M    494M  22%      86.5      83.5
  64     2767M     1966M    801M  29%      80.4      77.7
 128     4417M     2377M   2040M  46%      74.3      72.0


In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
N=[n for n in LADDER if base[n] and opt[n]]
f32=[base[n]['peak_rss_mb'] for n in N]; u8=[opt[n]['peak_rss_mb'] for n in N]
fig, ax = plt.subplots(1,2, figsize=(12,4.2))
ax[0].plot(N,f32,'o-',label='f32 (baseline)'); ax[0].plot(N,u8,'s-',label='u8 (optimized)')
for y,lab in [(2048,'2 GB'),(4096,'4 GB'),(6144,'6 GB'),(8192,'8 GB')]:
    ax[0].axhline(y,ls='--',c='grey',lw=.6); ax[0].text(N[0],y+40,lab,fontsize=7,c='grey')
ax[0].set_xscale('log',base=2); ax[0].set_xticks(N); ax[0].set_xticklabels(N)
ax[0].set_xlabel('images (N)'); ax[0].set_ylabel('peak host RSS (MB)')
ax[0].set_title('Host RAM vs image count'); ax[0].legend(); ax[0].grid(alpha=.3)
sv=[100*(base[n]['peak_rss_mb']-opt[n]['peak_rss_mb'])/base[n]['peak_rss_mb'] for n in N]
ax[1].bar([str(n) for n in N], sv, color='seagreen')
ax[1].set_xlabel('images (N)'); ax[1].set_ylabel('RSS saved (%)')
ax[1].set_title('u8 store: RAM saved (grows with data footprint)')
for i,v in enumerate(sv): ax[1].text(i,v+0.3,f'{v:.0f}%',ha='center',fontsize=8)
plt.tight_layout(); plt.savefig('/content/OpenSplat/linux_research/measurements/ram_plot.png',dpi=110)
plt.show(); print("u8 is bit-identical to f32 (verified by ctest test_cv_utils) -> 0 quality loss")

u8 is bit-identical to f32 (verified by ctest test_cv_utils) -> 0 quality loss


## 6. VRAM — the `--max-splats` cap

Densification grows the gaussian count (and VRAM) without bound. `--max-splats` (or the
auto cap derived from the VRAM budget) pauses growth at a budget. Measured ~**1810
bytes/gaussian** on this box (matches the README's "~2 GB per million").

In [8]:
import json, subprocess, pathlib
REPO=pathlib.Path('/content/OpenSplat')
def cap_run(args, tag):
    p=REPO/'linux_research'/'measurements'/'optimized'/f'{tag}.json'
    if not p.exists():
        subprocess.run(f"python3 scripts/bench_resource.py --bin ./output/opensplat "
                       f"--dataset data/chunks/drjohnson_32 --extra-args '-n 3000 {args}' "
                       f"--label {tag} --out {p} --poll-ms 30", shell=True, cwd=REPO, capture_output=True)
    return json.load(open(p))
unc=cap_run('','cap_uncapped'); cap=cap_run('--max-splats 60000','cap_60k')
for d,lab in [(unc,'uncapped'),(cap,'--max-splats 60000')]:
    print(f"{lab:18s} gaussians={d['gaussians_final']:>8}  peak_vram={d['peak_vram_mb']}MB  wall={d['wall_s']}s")
dg=unc['gaussians_final']-cap['gaussians_final']; dv=unc['peak_vram_mb']-cap['peak_vram_mb']
print(f"\nMeasured ~{dv*1e6/dg:.0f} bytes/gaussian (dVRAM {dv}MB / dGaussians {dg})")

uncapped           gaussians=  208929  peak_vram=954MB  wall=31.342s
--max-splats 60000 gaussians=   66449  peak_vram=696MB  wall=27.548s

Measured ~1811 bytes/gaussian (dVRAM 258MB / dGaussians 142480)


## 6b. GPU utilization & CUDA-specific optimizations

Three things make this a **CUDA** effort, not just a Linux one:

1. **The GPU is genuinely the bottleneck.** We sample `nvidia-smi utilization.gpu` while
   training — the A100 runs at **~67 % mean / ~88 % peak**, i.e. the loop is GPU-bound.
2. **VRAM bounding (`--max-splats`)** is the GPU-memory optimization: it caps gaussian count,
   and the cap is **auto-derived from the VRAM `nvidia-smi` reports**. Measured ~1899 B/gaussian.
3. **Fast-math kernels** (`-DOPENSPLAT_USE_FAST_MATH=ON`) were built and A/B-tested — **no
   measurable speedup** at this scale, so they stay **opt-in, not default** (an honest negative).
4. **On-GPU image conversion** (`getImageToDevice`) moves the u8→f32 convert onto the device and
   shrinks the host→device copy 4× under the u8 store.

In [9]:
import json, subprocess, pathlib
REPO = pathlib.Path('/content/OpenSplat')
# GPU utilization during a real training run (evidence the GPU is used + is the bottleneck)
g = REPO/'linux_research'/'measurements'/'optimized'/'gpu_default.json'
if not g.exists():
    subprocess.run("python3 scripts/bench_resource.py --bin ./output/opensplat "
                   "--dataset data/chunks/drjohnson_32 --extra-args '-n 2000' --label gpu_default "
                   f"--out {g} --poll-ms 25", shell=True, cwd=REPO, capture_output=True)
gd = json.load(open(g))
print(f"GPU utilization during training (drjohnson_32, -n2000):")
print(f"   mean {gd['gpu_util_mean_pct']}%  peak {gd['gpu_util_peak_pct']}%  "
      f"=> workload is GPU-BOUND | {gd['iters_per_s']} it/s | peak VRAM {gd['peak_vram_mb']} MB")
cuda = json.load(open(REPO/'linux_research'/'measurements'/'cuda_optimization_summary.json'))
print("\\nFast-math A/B (it/s):", cuda['fastmath_kernels']['default_it_s'], "default vs",
      cuda['fastmath_kernels']['fastmath_it_s'], "fast-math ->", cuda['fastmath_kernels']['verdict'])
vc = cuda['vram_cap_optimization']
print(f"\\nVRAM cap (CUDA memory optimization): uncapped {vc['uncapped']['gaussians']} g / "
      f"{vc['uncapped']['peak_vram_mb']}MB  ->  --max-splats 60000 {vc['capped_60000']['gaussians']} g / "
      f"{vc['capped_60000']['peak_vram_mb']}MB  (~{vc['bytes_per_gaussian_measured_MiB_basis']} B/gaussian)")

GPU utilization during training (drjohnson_32, -n2000):
   mean 66.7%  peak 88%  => workload is GPU-BOUND | 101.542 it/s | peak VRAM 702 MB
\nFast-math A/B (it/s): [100.8, 101.6] default vs [101.2, 101.3] fast-math -> NO measurable speedup at ~75k gaussians/1.17MP on A100 (within run-to-run noise); quality within atomicAdd nondeterminism band
\nVRAM cap (CUDA memory optimization): uncapped 208929 g / 954MB  ->  --max-splats 60000 66449 g / 696MB  (~1899 B/gaussian)


In [10]:
import matplotlib.pyplot as plt, json, pathlib
REPO=pathlib.Path('/content/OpenSplat')
cuda=json.load(open(REPO/'linux_research'/'measurements'/'cuda_optimization_summary.json'))
gd=json.load(open(REPO/'linux_research'/'measurements'/'optimized'/'gpu_default.json'))
fig,ax=plt.subplots(1,2,figsize=(11,4))
# GPU util gauge-ish bar
ax[0].bar(['mean','peak'],[gd['gpu_util_mean_pct'],gd['gpu_util_peak_pct']],color=['#4c72b0','#dd8452'])
ax[0].set_ylim(0,100); ax[0].set_ylabel('GPU utilization (%)'); ax[0].set_title('A100 utilization during training\\n(GPU-bound)')
for i,v in enumerate([gd['gpu_util_mean_pct'],gd['gpu_util_peak_pct']]): ax[0].text(i,v+1,f'{v}%',ha='center')
# VRAM cap before/after
vc=cuda['vram_cap_optimization']
ax[1].bar(['uncapped','--max-splats\\n60000'],[vc['uncapped']['peak_vram_mb'],vc['capped_60000']['peak_vram_mb']],color=['#c44e52','#55a868'])
ax[1].set_ylabel('peak VRAM (MB)'); ax[1].set_title('VRAM bounded by gaussian cap')
for i,v in enumerate([vc['uncapped']['peak_vram_mb'],vc['capped_60000']['peak_vram_mb']]): ax[1].text(i,v+5,f'{v}MB',ha='center')
plt.tight_layout(); plt.savefig(str(REPO/'linux_research'/'measurements'/'cuda_plot.png'),dpi=110); plt.show()

## 7. Tests — unit + regression + integration (incl. CUDA)

The suite covers the pure contract logic (`test_resource`, `test_regression_resource`),
the u8 bit-identity (`test_cv_utils`), the CPU pipeline (`test_integration`), and the real
**CUDA** path + `--max-splats` cap (`test_integration_cuda`, gated on a GPU being present).

In [11]:
import subprocess, torch, pathlib
REPO=pathlib.Path('/content/OpenSplat')
torch_cmake=pathlib.Path(torch.__file__).parent/'share'/'cmake'
subprocess.run(f"cmake -S {REPO} -B build -DOPENSPLAT_BUILD_TESTS=ON -DCMAKE_PREFIX_PATH={torch_cmake}",
               shell=True, cwd=REPO, capture_output=True)
subprocess.run("cmake --build build -j$(nproc) --target opensplat_all_tests", shell=True, cwd=REPO, capture_output=True)
r=subprocess.run("ctest --test-dir build --output-on-failure", shell=True, cwd=REPO, text=True, capture_output=True)
print(r.stdout[-1500:]); print("STDERR tail:", r.stderr[-300:])

Internal ctest changing into directory: /content/OpenSplat/build
Test project /content/OpenSplat/build
      Start  1: test_utils
 1/11 Test  #1: test_utils .......................   Passed    0.00 sec
      Start  2: test_tensor_math
 2/11 Test  #2: test_tensor_math .................   Passed    0.56 sec
      Start  3: test_ssim
 3/11 Test  #3: test_ssim ........................   Passed    0.60 sec
      Start  4: test_optim_scheduler
 4/11 Test  #4: test_optim_scheduler .............   Passed    0.56 sec
      Start  5: test_kdtree
 5/11 Test  #5: test_kdtree ......................   Passed    0.57 sec
      Start  6: test_cv_utils
 6/11 Test  #6: test_cv_utils ....................   Passed    0.74 sec
      Start  7: test_resource
 7/11 Test  #7: test_resource ....................   Passed    0.00 sec
      Start  8: test_regression_math
 8/11 Test  #8: test_regression_math .............   Passed    0.57 sec
      Start  9: test_regression_resource
 9/11 Test  #9: test_regression_

## 8. Summary

- **u8 image store** — up to **46 % less host RAM** at N=128 (4417 → 2377 MB), bit-identical, ~3 % slower; savings grow with the dataset's data footprint. Hot-path conversion moved on-GPU (`getImageToDevice`).
- **`--max-splats` cap** — bounds gaussian count → bounds VRAM (~1810 B/gaussian measured); auto-derived from the VRAM budget with 15 % headroom.
- **Process contract** — auto-detects the machine and picks a route; fully overridable (CLI > env > JSON > profile > auto) with printed provenance.
- **Quality guard** — `min-render-px` (default 400) warns before the over-downscaling divergence cliff documented in `linux_research/findings/03_quality_cliff_downscale.md`.

**CUDA-specific (before → after):**
- GPU is **GPU-bound** at ~67 % mean / 88 % peak A100 utilization (measured, not assumed).
- **VRAM**: unbounded densification 208,929 g / **954 MB** → `--max-splats` 66,449 g / **696 MB**; cap auto-derived from `nvidia-smi` VRAM (~1899 B/gaussian).
- **fast-math kernels**: built + A/B-tested → no measurable speedup here → kept **opt-in** (honest negative).
- **H2D**: u8→f32 conversion moved on-GPU (`getImageToDevice`), 4× smaller host→device copy under u8 store.

Full write-ups, Mermaid diagrams, methodology and the research roadmap are in **`linux_research/`**.
The RAM-budget harness `scripts/bench_resource.py` uses kernel **cgroup `memory.max`** when
writable and falls back to a **peak-RSS watchdog** (Colab's cgroup fs is read-only).